# 💅 Nail-Bed Anemia Classifier — Training Notebook (Google Colab → Raspberry Pi 5)

This notebook trains a model that looks at a **photo of a fingernail bed** and says **Anemic** or **Not anemic**.

**How it works (in plain words):** we take a model that Google already trained on millions of everyday photos (**EfficientNet-B0**) and *re-teach* only its last part to spot nail-bed pallor. This is called **transfer learning**, and it works well even with a few thousand images. At the end, the model is converted to **TensorFlow Lite** (`.tflite`), a small, fast format that runs on the Raspberry Pi 5.

### How to use it
1. In Colab: **Runtime → Change runtime type → T4 GPU → Save**.
2. **Runtime → Run all**. That's it — two public datasets are **downloaded automatically** (Step 3):
   - 🇬🇭 **Ghana** fingernail dataset — darker skin (children ≤ 5 years)
   - 🇷🇺 **Yakimov et al. 2024** (Scientific Data) — lighter skin (adults, lab-measured hemoglobin)
3. *(Optional)* add your own photos from Google Drive with `MY_DATA_PATH` in **Step 2**.
4. Training takes roughly 15–40 minutes.
5. At the end you get a `.zip` with the model + a ready-to-run Raspberry Pi script.

### Things this notebook does for you (that beginners usually miss)
- **No cheating in the test score:** many nail datasets contain *augmented copies* of the same photo. If copies land in both training and test, accuracy looks fake-high. The notebook finds near-duplicate photos (even rotated/flipped ones) and keeps them together in the same split.
- **Does not destroy the colour signal:** anemia shows up as *pale colour*, so augmentation never changes hue or saturation, only position, small brightness and contrast changes.
- **Fairness check across skin tones / datasets:** if you tell it which group each image belongs to, it reports accuracy separately for each group, so you can *see* whether it works equally on lighter and darker skin.
- **Screening threshold:** it picks a decision threshold that favours catching anemic people (sensitivity) over raw accuracy.

> ⚠️ This is a **screening prototype**, not a medical device. Always confirm with a hemoglobin blood test.

## Step 1 — Setup
Installs two small libraries and checks that a GPU is available.

In [ ]:
!pip install -q imagehash pillow-heif ai-edge-litert

import os, json, glob, random, shutil, zipfile, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash
import tensorflow as tf
keras = tf.keras

try:
    import pillow_heif
    pillow_heif.register_heif_opener()      # lets us read iPhone .HEIC photos
except Exception:
    pass

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NO GPU FOUND -> Runtime > Change runtime type > T4 GPU (training will be very slow on CPU)")

## Step 2 — Settings ✏️ (edit this cell)

| Setting | What it means |
|---|---|
| `USE_PUBLIC_DATASETS` | `True` = download the Ghana + Yakimov datasets automatically. |
| `HB_THRESHOLD_GL` | Hemoglobin cut-off (g/L) used to label the Yakimov photos as anemic. 120 g/L is the WHO cut-off for adult women (men: 130). |
| `MY_DATA_PATH` | *Optional* — your own photos in Google Drive (a **folder** or **.zip**), added on top. Leave `""` if none. |
| `METADATA_CSV` | *Optional* CSV with extra info per image (person ID, skin-tone group). Leave `""` if you don't have one. |
| `IMG_SIZE` | Images are resized to this. 224 is standard; keep it. |
| `EPOCHS_HEAD`, `EPOCHS_FINETUNE` | Max training rounds for the two training phases. Training stops early automatically when it stops improving. |
| `TARGET_SENSITIVITY` | How many anemic people (out of 1.0) we want to catch. 0.85 = catch 85%. Higher = more false alarms. |
| `DEDUP_HAMMING` | How similar two photos must be to count as "the same photo". Leave at 4. |

In [ ]:
# ---- DATA ----
USE_PUBLIC_DATASETS = True    # auto-download Ghana (darker skin) + Yakimov 2024 (lighter skin)
HB_THRESHOLD_GL     = 120     # Yakimov photos with Hb below this (g/L) are labelled anemic
MY_DATA_PATH        = ""      # optional extra data, e.g. "/content/drive/MyDrive/my_nail_photos.zip"
METADATA_CSV        = ""      # optional CSV for MY data, e.g. "/content/drive/MyDrive/my_metadata.csv"

# ---- MODEL / TRAINING ----
IMG_SIZE        = 224
BATCH_SIZE      = 32
EPOCHS_HEAD     = 15      # phase 1: train only the new last layer
EPOCHS_FINETUNE = 25      # phase 2: gently adjust the top of EfficientNet too
FINETUNE_LAYERS = 60      # how many of the top EfficientNet layers to unfreeze in phase 2
WEIGHTS         = "imagenet"   # pretrained starting point (keep this)

# ---- SPLIT / EVALUATION ----
TEST_FRACTION      = 0.15
VAL_FRACTION       = 0.15
TARGET_SENSITIVITY = 0.85
DEDUP_HAMMING      = 4

# ---- OUTPUT ----
OUTPUT_DIR       = "/content/anemia_model_output"
SAVE_TO_DRIVE    = True     # copies the trained model to Google Drive (asks for Drive access)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/anemia_model_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Step 3 — Get the data

### Public datasets (downloaded automatically)
| Dataset | People | Skin tone | Labels | Licence |
|---|---|---|---|---|
| **Ghana fingernails** (Asare et al., Mendeley Data 2022) | children ≤ 5 yrs, Ghana | darker | anemic / non-anemic folders. 710 originals, augmented to ~4,260 copies | CC BY 4.0 |
| **Yakimov et al., Scientific Data 2024** | 250 adults, Russia | lighter | lab hemoglobin (g/L) → labelled with `HB_THRESHOLD_GL` | CC0 |

For Yakimov, each photo shows a whole hand on a white background with the nail positions given, so the notebook **cuts out each fingernail** automatically (3 nails per person) and keeps all nails of one person together in the same split.

**Honest limits:** neither dataset contains Indian adults, and the Ghana set is young children only. Your model will be more reliable on your own users if you add your own labelled photos (see below). The **per-source results** in Step 8 show whether it works on both groups.

### Adding your own photos (optional)
Put them in Drive like this and set `MY_DATA_PATH` in Step 2 (a folder or a .zip):
```
my_nail_photos/
├── anemic/
└── non_anemic/
```
Folder names are matched loosely (`Anemic`, `anaemic`, `Non-Anemic`, `normal`, `healthy`...). File names containing the class (e.g. `anemic_12.jpg`) also work.

Optional `METADATA_CSV` for your photos (only `filename` is required):
```
filename,person_id,group
anemic/img001.jpg,P001,dark
non_anemic/img101.jpg,P042,light
```
- `person_id` → photos of the same person stay together, so the test truly measures *new* people.
- `group` → anything you want a separate accuracy report for (e.g. skin tone on the Monk scale).

Supported image types: jpg, png, bmp, webp, tiff, and iPhone HEIC.

In [ ]:
import subprocess, urllib.request

if IN_COLAB and (MY_DATA_PATH.startswith("/content/drive") or METADATA_CSV.startswith("/content/drive") or SAVE_TO_DRIVE):
    from google.colab import drive
    drive.mount("/content/drive")

BASE = "/content" if IN_COLAB else os.path.abspath(".")
DATA_ROOT = os.path.join(BASE, "data")         # every image used for training ends up in here
RAW_DIR = os.path.join(BASE, "raw_downloads")
shutil.rmtree(DATA_ROOT, ignore_errors=True)
os.makedirs(DATA_ROOT); os.makedirs(RAW_DIR, exist_ok=True)
AUTO_META = []                                 # person IDs we know for the public data

GHANA_URL   = "https://data.mendeley.com/public-files/datasets/2xx4j3kjg2/files/e3d97e08-0c09-45bc-bde6-68c30e98319b/file_downloaded"
YAKIMOV_URL = "https://ndownloader.figshare.com/files/49504041"

def download(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print("  already downloaded:", os.path.basename(dest)); return dest
    print("  downloading", os.path.basename(dest), "...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
            print(f"  done ({os.path.getsize(dest)/1e6:.1f} MB)"); return dest
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}"); time.sleep(5)
    raise RuntimeError(f"Could not download {url}. Download it in your browser, upload it to Google Drive, "
                       f"and use MY_DATA_PATH instead.")

def extract(archive, dest):
    os.makedirs(dest, exist_ok=True)
    with open(archive, "rb") as f:
        magic = f.read(4)
    if magic[:2] == b"PK":
        with zipfile.ZipFile(archive) as z: z.extractall(dest)
        return
    # RAR archive: use 7z or unrar (installed automatically if missing)
    if not (shutil.which("7z") or shutil.which("unrar")):
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    if shutil.which("unrar"):
        cmd = ["unrar", "x", "-o+", "-inul", archive, dest + "/"]
    else:
        cmd = ["7z", "x", "-y", f"-o{dest}", archive]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Could not extract {archive}:\n{r.stderr[-1000:]}")

def show_tree(root, max_lines=25):
    lines = 0
    for d, subdirs, files in sorted(os.walk(root)):
        subdirs.sort()
        rel = os.path.relpath(d, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        n_img = sum(os.path.splitext(f)[1].lower() in (".jpg", ".jpeg", ".png", ".bmp") for f in files)
        print("   " + "  " * depth + os.path.basename(d) + f"/  ({n_img} images)")
        lines += 1
        if lines >= max_lines:
            print("   ..."); break

if USE_PUBLIC_DATASETS:
    # ---- 1) Ghana fingernails (darker skin) ----
    print("[1/2] Ghana fingernail dataset")
    arc = download(GHANA_URL, os.path.join(RAW_DIR, "ghana_fingernails.rar"))
    extract(arc, os.path.join(DATA_ROOT, "ghana"))
    show_tree(os.path.join(DATA_ROOT, "ghana"))

    # ---- 2) Yakimov et al. 2024 (lighter skin): crop each nail from the hand photo ----
    print("\n[2/2] Yakimov et al. 2024 dataset")
    arc = download(YAKIMOV_URL, os.path.join(RAW_DIR, "yakimov_data.zip"))
    ydir = os.path.join(RAW_DIR, "yakimov")
    if not os.path.isdir(ydir):
        extract(arc, ydir)
    meta_path = glob.glob(os.path.join(ydir, "**", "metadata.csv"), recursive=True)[0]
    ymeta = pd.read_csv(meta_path)
    photo_index = {os.path.splitext(os.path.basename(p))[0]: p
                   for p in glob.glob(os.path.join(ydir, "**", "*.*"), recursive=True)
                   if os.path.splitext(p)[1].lower() in (".jpg", ".jpeg", ".png")}
    n_crops = 0
    for _, row in ymeta.iterrows():
        pid = str(row["PATIENT_ID"])
        if pid not in photo_index or pd.isna(row["HB_LEVEL_GperL"]):
            continue
        cls = "anemic" if float(row["HB_LEVEL_GperL"]) < HB_THRESHOLD_GL else "non_anemic"
        out_dir = os.path.join(DATA_ROOT, "yakimov", cls); os.makedirs(out_dir, exist_ok=True)
        with Image.open(photo_index[pid]) as im:
            hand = im.convert("RGB")
        for k, (top, left, bot, right) in enumerate(json.loads(row["NAIL_BOUNDING_BOXES"])):
            left, top = max(0, int(left)), max(0, int(top))
            right, bot = min(hand.width, int(right)), min(hand.height, int(bot))
            if right - left < 5 or bot - top < 5:
                continue
            fn = f"p{pid}_nail{k+1}.png"
            hand.crop((left, top, right, bot)).save(os.path.join(out_dir, fn))
            AUTO_META.append({"filename": f"yakimov/{cls}/{fn}", "person_id": f"yakimov_{pid}"})
            n_crops += 1
    print(f"  cropped {n_crops} nails from {ymeta['PATIENT_ID'].nunique()} people "
          f"(anemic = Hb < {HB_THRESHOLD_GL} g/L)")
    show_tree(os.path.join(DATA_ROOT, "yakimov"))

if MY_DATA_PATH:
    print("\nAdding your own data from", MY_DATA_PATH)
    mine = os.path.join(DATA_ROOT, "my_data")
    if MY_DATA_PATH.lower().endswith(".zip"):
        extract(MY_DATA_PATH, mine)
    else:
        shutil.copytree(MY_DATA_PATH, mine)
    show_tree(mine)

assert USE_PUBLIC_DATASETS or MY_DATA_PATH, "No data: set USE_PUBLIC_DATASETS = True or MY_DATA_PATH in Step 2."
print("\nAll training images are in:", DATA_ROOT)

### Scan, label and pre-process every image
Each image is opened once, resized to 224×224 (keeping its shape, black padding around), and kept in memory. We also compute a "fingerprint" (perceptual hash) of every image to detect duplicates.

In [ ]:
def letterbox(img, size=224):
    """Resize keeping the aspect ratio, then pad with black to size x size.
    The SAME function is used on the Raspberry Pi, so training and deployment match."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    s = size / max(w, h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    img = img.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    canvas.paste(img, ((size - nw) // 2, (size - nh) // 2))
    return canvas


IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".heic", ".heif"}
ANEMIC_WORDS = ("anemic", "anaemic", "anemia", "anaemia")

def class_from_name(name):
    n = "".join(c for c in name.lower() if c.isalpha())
    if n in ("normal", "healthy"):
        return 0
    for k in ANEMIC_WORDS:
        if n.startswith("non" + k) or n.startswith("not" + k):
            return 0
    for k in ANEMIC_WORDS:
        if n.startswith(k):
            return 1
    return None

def label_and_source(path):
    parts = os.path.relpath(path, DATA_ROOT).split(os.sep)
    source = parts[0] if len(parts) > 1 else "main"      # top folder = which dataset it came from
    for name in reversed(parts[:-1]):                     # 1) class from a folder name
        c = class_from_name(name)
        if c is not None:
            return c, source
    stem = os.path.splitext(parts[-1])[0]                 # 2) class from the file name (e.g. "anemic_12.jpg")
    return class_from_name(stem), source

def dihedral_hashes(img):
    """phash of the image in all 8 rotations/flips -> catches rotated/flipped augmented copies."""
    out = []
    for flip in (False, True):
        base = ImageOps.mirror(img) if flip else img
        for rot in (0, 90, 180, 270):
            h = imagehash.phash(base.rotate(rot))
            out.append(np.packbits(h.hash.flatten()))
    return np.stack(out)          # (8, 8) uint8

def process(path):
    try:
        with Image.open(path) as im:
            img = letterbox(im, IMG_SIZE)
        return np.asarray(img, dtype=np.uint8), dihedral_hashes(img)
    except Exception as e:
        return None, str(e)

all_files = [p for p in glob.glob(os.path.join(DATA_ROOT, "**", "*"), recursive=True)
             if os.path.splitext(p)[1].lower() in IMG_EXTS and "__MACOSX" not in p
             and not os.path.basename(p).startswith(".")]
records, skipped_nolabel = [], 0
for p in sorted(all_files):
    lab, src = label_and_source(p)
    if lab is None:
        skipped_nolabel += 1
        continue
    records.append({"path": p, "relpath": os.path.relpath(p, DATA_ROOT), "label": lab, "source": src})
print(f"Found {len(all_files)} image files, {len(records)} with a recognised class folder"
      f" ({skipped_nolabel} skipped because their folder name wasn't anemic/non-anemic).")
assert len(records) > 0, "No labelled images found. Check the folder names (see Step 3)."

t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(ex.map(process, [r["path"] for r in records]))
imgs, hashes, keep = [], [], []
for r, (arr, h) in zip(records, results):
    if arr is None:
        print("  ! could not read", r["relpath"], "->", h)
        continue
    imgs.append(arr); hashes.append(h); keep.append(r)
df = pd.DataFrame(keep).reset_index(drop=True)
X = np.stack(imgs)                       # (N, 224, 224, 3) uint8
HASHES = np.stack(hashes)                # (N, 8, 8)
y = df["label"].values.astype(np.int32)  # 1 = anemic, 0 = not anemic
print(f"Loaded {len(df)} images in {time.time()-t0:.0f}s. Memory: {X.nbytes/1e9:.2f} GB")

### Attach the optional metadata (person ID, skin-tone group)

In [ ]:
df["person_id"] = ""
df["group"] = ""
meta_parts = [pd.DataFrame(AUTO_META)] if AUTO_META else []
if METADATA_CSV:
    m = pd.read_csv(METADATA_CSV)
    m.columns = [c.strip().lower() for c in m.columns]
    assert "filename" in m.columns, "metadata CSV needs a 'filename' column"
    m["filename"] = "my_data/" + m["filename"].astype(str).str.replace("\\", "/").str.lstrip("/")
    meta_parts.append(m)
if meta_parts:
    meta = pd.concat(meta_parts, ignore_index=True)
    meta["filename"] = meta["filename"].astype(str).str.replace("\\", "/")
    by_rel = meta.set_index("filename")
    by_base = meta.assign(b=meta["filename"].map(os.path.basename)).drop_duplicates("b").set_index("b")
    matched = 0
    for i, rel in enumerate(df["relpath"].str.replace("\\", "/")):
        row = by_rel.loc[rel] if rel in by_rel.index else (by_base.loc[os.path.basename(rel)] if os.path.basename(rel) in by_base.index else None)
        if row is None:
            continue
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        matched += 1
        if "person_id" in row and pd.notna(row["person_id"]):
            df.at[i, "person_id"] = str(row["person_id"])
        if "group" in row and pd.notna(row["group"]):
            df.at[i, "group"] = str(row["group"])
    print(f"Person / group info attached to {matched}/{len(df)} images")
else:
    print("No metadata - that's OK.")

has_groups = (df["group"] != "").any()
has_sources = df["source"].nunique() > 1

### Look at the data
Check the counts and **look at the sample photos** — are they all nail beds? Are both classes photographed the same way? (If anemic photos come from one camera and healthy ones from another, the model can learn the *camera* instead of anemia.)

In [ ]:
df["class"] = np.where(y == 1, "anemic", "not anemic")
print("Images per class:")
print(df["class"].value_counts().to_string(), "\n")
if has_sources:
    print("Images per source / class:")
    print(pd.crosstab(df["source"], df["class"]).to_string(), "\n")
if has_groups:
    print("Images per group / class:")
    print(pd.crosstab(df["group"].replace("", "(none)"), df["class"]).to_string(), "\n")
if has_sources:
    print("Note: if one source is mostly anemic and another mostly healthy, the model can learn 'which dataset'\n"
          "instead of 'anemia'. The per-source results in Step 8 reveal this.\n")
if len(df) < 300:
    print("⚠️ Fewer than 300 images: results will be unreliable. Try to collect more.")

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for row, cls in enumerate([1, 0]):
    idx = np.where(y == cls)[0]
    pick = np.random.choice(idx, min(8, len(idx)), replace=False)
    for ax in axes[row]:
        ax.axis("off")
    for ax, i in zip(axes[row], pick):
        ax.imshow(X[i]); ax.set_title(f"{df['class'][i]}\n{df['source'][i]}", fontsize=8)
plt.suptitle("Random samples (top: anemic, bottom: not anemic)"); plt.tight_layout(); plt.show()

## Step 4 — Find duplicates & split into train / validation / test

- **Train** (~70%): the model learns from these.
- **Validation** (~15%): used during training to decide when to stop and to choose the threshold.
- **Test** (~15%): never seen during training — this is your honest final score.

Near-duplicate photos (and photos of the same person, if you gave `person_id`) are kept in the **same** split.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def build_groups(max_dist):
    n = len(df)
    parent = np.arange(n)
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    if max_dist >= 0:
        H0 = HASHES[:, 0, :]                      # original orientation
        for i in range(n - 1):
            d = POPCOUNT[np.bitwise_xor(HASHES[i + 1:], H0[i])].sum(-1).min(-1)
            for j in np.nonzero(d <= max_dist)[0]:
                union(i, i + 1 + j)
    pid = df["person_id"].values
    first = {}
    for i, p in enumerate(pid):
        if p:
            if p in first: union(first[p], i)
            else: first[p] = i
    return np.array([find(i) for i in range(n)])

dist = DEDUP_HAMMING
while True:
    groups = build_groups(dist)
    sizes = pd.Series(groups).value_counts()
    if sizes.iloc[0] <= max(10, 0.05 * len(df)) or dist < 0:
        break
    print(f"Duplicate threshold {dist} merged too many different photos (biggest cluster {sizes.iloc[0]}); tightening...")
    dist -= 1

n_clusters = (sizes > 1).sum()
print(f"Near-duplicate threshold used: {dist}")
print(f"{len(df)} images -> {len(sizes)} independent groups ({n_clusters} groups contain duplicates/same person).")
mixed = [g for g in sizes.index[sizes > 1] if len(set(y[groups == g])) > 1]
if mixed:
    print(f"⚠️ {len(mixed)} duplicate groups contain BOTH labels - the same photo is labelled anemic and not anemic somewhere. Check your data.")
df["dup_group"] = groups

def group_split(idx, frac, seed):
    k = max(2, int(round(1 / frac)))
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(np.zeros(len(idx)), y[idx], groups[idx]))
    return idx[a], idx[b]

all_idx = np.arange(len(df))
trainval_idx, test_idx = group_split(all_idx, TEST_FRACTION, SEED)
train_idx, val_idx = group_split(trainval_idx, VAL_FRACTION / (1 - TEST_FRACTION), SEED)

for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name:5s}: {len(idx):5d} images | anemic {int(y[idx].sum()):4d} | not anemic {int((1 - y[idx]).sum()):4d}")
    assert len(set(y[idx])) == 2, f"The {name} split has only one class - you need more data."
assert not (set(groups[train_idx]) & set(groups[test_idx])), "leak between train and test!"
print("✅ No duplicate / same-person leakage between splits.")

## Step 5 — Data pipeline & augmentation

**Augmentation** = showing the model slightly changed copies of training images (flipped, rotated, zoomed, a bit brighter/darker) so it learns the *nail colour* and not the exact photo.
We deliberately **do not** change hue or saturation — that's exactly the pallor signal we want the model to learn.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.08, fill_mode="constant"),
    keras.layers.RandomZoom((-0.15, 0.15), fill_mode="constant"),
    keras.layers.RandomTranslation(0.08, 0.08, fill_mode="constant"),
    keras.layers.RandomBrightness(0.06, value_range=(0, 255)),
    keras.layers.RandomContrast(0.10),
], name="augment")

def make_ds(idx, training):
    idx = np.array(idx)
    def gen():
        order = np.random.permutation(idx) if training else idx
        for i in order:
            yield X[i], np.float32(y[i])
    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.uint8), tf.TensorSpec((), tf.float32)))
    ds = ds.batch(BATCH_SIZE).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda a, b: (tf.clip_by_value(augment(a, training=True), 0, 255), b), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_idx, True)
val_ds = make_ds(val_idx, False)

# Class weights: if one class has fewer images, its mistakes count more
n_pos, n_neg = int(y[train_idx].sum()), int((1 - y[train_idx]).sum())
class_weight = {0: len(train_idx) / (2 * n_neg), 1: len(train_idx) / (2 * n_pos)}
print("Class weights:", {k: round(v, 2) for k, v in class_weight.items()})

# Show what augmented images look like
xb, yb = next(iter(train_ds))
plt.figure(figsize=(16, 4))
for i in range(min(8, len(xb))):
    plt.subplot(1, 8, i + 1); plt.imshow(xb[i].numpy().astype("uint8")); plt.axis("off")
    plt.title("anemic" if yb[i] == 1 else "not anemic", fontsize=8)
plt.suptitle("Augmented training images"); plt.show()

## Step 6 — Build the model
**EfficientNet-B0** (pretrained) → average pooling → dropout → 1 output neuron giving the **probability of anemia** (0 to 1).
EfficientNet-B0 is small (~4M parameters, ~16 MB) and runs comfortably on a Raspberry Pi 5. It takes raw pixel values 0–255, so no extra normalisation is needed on the Pi.

In [ ]:
def build_model():
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS,
                                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
    for layer in base.layers:
        layer.trainable = False
    x = keras.layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = keras.layers.Dropout(0.3, name="dropout")(x)
    out = keras.layers.Dense(1, activation="sigmoid", name="anemia_prob")(x)
    return keras.Model(base.input, out, name="nail_anemia_effb0"), base

def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc"),
                       keras.metrics.BinaryAccuracy(name="accuracy"),
                       keras.metrics.Recall(name="sensitivity"),
                       keras.metrics.Precision(name="precision")])

model, base = build_model()
compile_model(model, 1e-3)
print(f"Total parameters: {model.count_params():,}")

## Step 7 — Train

**Phase 1:** EfficientNet is frozen; only the new last layer learns (fast, safe).
**Phase 2 (fine-tuning):** the top layers of EfficientNet are unlocked and adjusted with a very small learning rate so the model becomes sensitive to subtle nail colours.

Training automatically stops when the validation AUC stops improving and keeps the best version.

> **AUC** = how well the model ranks anemic above non-anemic (0.5 = guessing, 1.0 = perfect).

In [ ]:
best_path = os.path.join(OUTPUT_DIR, "best_model.keras")
def callbacks(patience):
    return [
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.3, patience=max(2, patience // 2), min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(best_path, monitor="val_auc", mode="max", save_best_only=True),
    ]

print("===== Phase 1: training the new head =====")
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                  class_weight=class_weight, callbacks=callbacks(5), verbose=2)

print("\n===== Phase 2: fine-tuning the top of EfficientNet =====")
for layer in base.layers[-FINETUNE_LAYERS:]:
    if not isinstance(layer, keras.layers.BatchNormalization):   # BatchNorm stays frozen (important for small datasets)
        layer.trainable = True
compile_model(model, 2e-5)
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
                  class_weight=class_weight, callbacks=callbacks(6), verbose=2)

In [ ]:
def plot_history(h1, h2):
    hist = {k: h1.history.get(k, []) + h2.history.get(k, []) for k in h1.history}
    n1 = len(h1.history["loss"])
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for ax, key in zip(axes, ["loss", "auc", "sensitivity"]):
        ax.plot(hist[key], label="train"); ax.plot(hist["val_" + key], label="validation")
        ax.axvline(n1 - 0.5, color="gray", ls="--", lw=1); ax.set_title(key); ax.set_xlabel("epoch"); ax.legend()
    plt.suptitle("Training curves (dashed line = start of fine-tuning)"); plt.show()
plot_history(hist1, hist2)
print("If 'train' keeps improving while 'validation' gets worse, the model is memorising (overfitting) - more data helps most.")

## Step 8 — Choose the decision threshold & test the model

The model outputs a probability. We choose the cut-off on the **validation** set so that at least `TARGET_SENSITIVITY` of anemic people are flagged, then report honest results on the untouched **test** set.

- **Sensitivity** = of the truly anemic people, how many we caught.
- **Specificity** = of the healthy people, how many we correctly cleared.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

def predict_probs(idx):
    return model.predict(make_ds(idx, False), verbose=0).ravel()

p_val, p_test = predict_probs(val_idx), predict_probs(test_idx)
y_val, y_test = y[val_idx], y[test_idx]

fpr, tpr, thr = roc_curve(y_val, p_val)
thr = np.clip(thr, 0, 1)
ok = np.where(tpr >= TARGET_SENSITIVITY)[0]
if len(ok):
    best = ok[np.argmin(fpr[ok])]
    how = f"sensitivity >= {TARGET_SENSITIVITY} on validation"
else:
    best = np.argmax(tpr - fpr)
    how = "best balance (Youden's J)"
THRESHOLD = float(thr[best])
print(f"Chosen threshold: {THRESHOLD:.3f}  ({how})")

def metrics(y_true, prob, t):
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(y_true, prob) if len(set(y_true)) > 1 else float("nan")
    r = {"accuracy": (tp + tn) / len(y_true),
         "sensitivity": tp / (tp + fn) if tp + fn else float("nan"),
         "specificity": tn / (tn + fp) if tn + fp else float("nan"),
         "auc": auc}
    return {"n": int(len(y_true)), **{k: round(float(v), 3) for k, v in r.items()}}

test_metrics = metrics(y_test, p_test, THRESHOLD)
print("\n===== TEST SET RESULTS (images the model never saw) =====")
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v:.3f}" if isinstance(v, float) else f"  {k:12s}: {v}")
print(f"  (at the default 0.5 threshold: {metrics(y_test, p_test, 0.5)})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, (p_test >= THRESHOLD).astype(int)),
                       display_labels=["not anemic", "anemic"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Test confusion matrix")
tf_, tt_, _ = roc_curve(y_test, p_test)
axes[1].plot(tf_, tt_, label=f"AUC = {test_metrics['auc']:.3f}"); axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False positive rate (1 - specificity)"); axes[1].set_ylabel("Sensitivity"); axes[1].set_title("Test ROC curve"); axes[1].legend()
plt.tight_layout(); plt.show()

### Fairness check — does it work equally well for every group?
This is how you **prove** the model works on both lighter and darker skin. Results are shown per **source folder** and per **group** from your metadata CSV.
A big gap between groups means that group needs more training images.

In [ ]:
def per_group_report(col):
    rows = []
    for g in sorted(df[col].unique()):
        if g == "": continue
        m = df[col].values[test_idx] == g
        if m.sum() == 0: continue
        r = metrics(y_test[m], p_test[m], THRESHOLD); r[col] = g
        r["anemic_in_test"] = int(y_test[m].sum())
        rows.append(r)
    if rows:
        out = pd.DataFrame(rows).set_index(col)[["n", "anemic_in_test", "accuracy", "sensitivity", "specificity", "auc"]]
        print(f"\nTest results per {col}:"); print(out.round(3).to_string())
        small = out[out["n"] < 30].index.tolist()
        if small:
            print(f"  (groups with < 30 test images are too small to trust: {small})")

if has_sources:
    per_group_report("source")
if has_groups:
    per_group_report("group")
if not (has_sources or has_groups):
    print("No groups available. To check skin-tone fairness, add a 'group' column via METADATA_CSV,\n"
          "or put images from different populations/datasets in separate source folders.")

### Where is the model looking? (Grad-CAM)
Red areas = what influenced the decision most. They should be **on the nail bed**, not on the background, padding or image corners. If it looks at the background, the model has learned a shortcut.

In [ ]:
try:
    grad_model = keras.Model(model.inputs, [model.get_layer("top_activation").output, model.output])
    def gradcam(img):
        x = tf.convert_to_tensor(img[None].astype("float32"))
        with tf.GradientTape() as tape:
            conv, pred = grad_model(x, training=False)
            score = pred[:, 0]
        grads = tape.gradient(score, conv)
        w = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1))
        cam = cam / (tf.reduce_max(cam) + 1e-8)
        cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE))[..., 0]
        return cam.numpy(), float(pred[0, 0])
    pick = np.random.choice(test_idx, min(8, len(test_idx)), replace=False)
    plt.figure(figsize=(18, 5))
    for k, i in enumerate(pick):
        cam, p = gradcam(X[i])
        plt.subplot(2, 8, k + 1); plt.imshow(X[i]); plt.axis("off")
        plt.title(f"true: {'anemic' if y[i] else 'normal'}\np={p:.2f}", fontsize=8)
        plt.subplot(2, 8, k + 9); plt.imshow(X[i]); plt.imshow(cam, cmap="jet", alpha=0.45); plt.axis("off")
    plt.suptitle("Grad-CAM on test images (bottom row: where the model looks)"); plt.show()
except Exception as e:
    print("Grad-CAM skipped:", e)

## Step 9 — Export for Raspberry Pi 5 (TensorFlow Lite)
Two versions are created and both are re-tested here so you know they still work after conversion:
- `anemia_model_float32.tflite` — most accurate, ~16 MB. **Start with this one.**
- `anemia_model_int8.tflite` — ~4–5 MB and faster; use it only if its test results below match the float32 one.

In [ ]:
model.save(os.path.join(OUTPUT_DIR, "anemia_model.keras"))
saved_dir = os.path.join(OUTPUT_DIR, "saved_model")
shutil.rmtree(saved_dir, ignore_errors=True)
if hasattr(model, "export"):
    try:
        model.export(saved_dir, verbose=False)
    except TypeError:
        model.export(saved_dir)
else:
    tf.saved_model.save(model, saved_dir)

def convert(int8=False):
    conv = tf.lite.TFLiteConverter.from_saved_model(saved_dir)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        rep_idx = np.random.choice(train_idx, min(300, len(train_idx)), replace=False)
        def rep():
            for i in rep_idx:
                yield [X[i][None].astype(np.float32)]
        conv.representative_dataset = rep      # input/output stay float32 -> same Pi code
    return conv.convert()

tflite_files = {}
for name, int8 in [("float32", False), ("int8", True)]:
    try:
        blob = convert(int8)
        path = os.path.join(OUTPUT_DIR, f"anemia_model_{name}.tflite")
        open(path, "wb").write(blob)
        tflite_files[name] = path
        print(f"Saved {path}  ({len(blob)/1e6:.1f} MB)")
    except Exception as e:
        print(f"Could not create {name} model: {e}")

try:
    from ai_edge_litert.interpreter import Interpreter as LiteInterpreter   # same runtime as on the Pi
except ImportError:
    LiteInterpreter = tf.lite.Interpreter

def tflite_probs(path, idx):
    it = LiteInterpreter(model_path=path, num_threads=2); it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    probs, t0 = [], time.perf_counter()
    for i in idx:
        it.set_tensor(inp["index"], X[i][None].astype(np.float32)); it.invoke()
        probs.append(float(np.squeeze(it.get_tensor(out["index"]))))
    return np.array(probs), (time.perf_counter() - t0) / len(idx) * 1000

print("\nRe-testing the converted models on the test set:")
print(f"  keras   : {metrics(y_test, p_test, THRESHOLD)}")
tflite_results = {}
for name, path in tflite_files.items():
    p, ms = tflite_probs(path, test_idx)
    tflite_results[name] = metrics(y_test, p, THRESHOLD)
    print(f"  {name:8s}: {tflite_results[name]}   max prob diff vs keras = {np.abs(p - p_test).max():.4f}   (~{ms:.0f} ms/img on Colab CPU)")

## Step 10 — Save everything + Raspberry Pi script
Creates `anemia_pi_package.zip` (~21 MB) containing the two `.tflite` models, `model_config.json` (threshold etc.) and `predict_pi.py`. It is downloaded to your computer and also copied to your Google Drive.

**On the Raspberry Pi 5:** unzip, then follow the instructions at the top of `predict_pi.py`:
```
python3 -m venv --system-site-packages ~/anemia-env
source ~/anemia-env/bin/activate
pip install ai-edge-litert pillow numpy
python predict_pi.py --image some_nail_photo.jpg
python predict_pi.py --camera
```

In [ ]:
config = {
    "model": "EfficientNetB0 transfer learning",
    "img_size": IMG_SIZE,
    "threshold": THRESHOLD,
    "output": "sigmoid probability of anemia (1 = anemic)",
    "input": "float32 RGB pixels 0-255, shape [1, H, W, 3]; letterbox resize with black padding (see predict_pi.py)",
    "target_sensitivity": TARGET_SENSITIVITY,
    "test_metrics_keras": {k: float(v) for k, v in test_metrics.items()},
    "test_metrics_tflite": {n: {k: float(v) for k, v in m.items()} for n, m in tflite_results.items()},
    "n_images": {"train": int(len(train_idx)), "val": int(len(val_idx)), "test": int(len(test_idx))},
    "trained_on": time.strftime("%Y-%m-%d %H:%M"),
    "tensorflow_version": tf.__version__,
}
with open(os.path.join(OUTPUT_DIR, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)

PI_SCRIPT = r'''#!/usr/bin/env python3
"""
Nail-bed anemia screening - Raspberry Pi 5 inference script.

SETUP (once, on the Pi):
    sudo apt install -y python3-picamera2        # only if you use the Pi camera
    python3 -m venv --system-site-packages ~/anemia-env
    source ~/anemia-env/bin/activate
    pip install ai-edge-litert pillow numpy
    # if ai-edge-litert has no wheel for your Python, use: pip install tensorflow

FILES needed in the same folder as this script:
    anemia_model_float32.tflite   (or anemia_model_int8.tflite)
    model_config.json

USAGE:
    python predict_pi.py --image nail.jpg
    python predict_pi.py --camera              # capture one photo from the Pi camera
    python predict_pi.py --image nail.jpg --model anemia_model_int8.tflite

This is a SCREENING prototype, not a medical diagnosis.
"""
import argparse, json, os, time
import numpy as np
from PIL import Image, ImageOps

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    try:
        from tflite_runtime.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter

HERE = os.path.dirname(os.path.abspath(__file__))


def letterbox(img, size=224):
    """Resize keeping the aspect ratio, then pad with black to size x size.
    The SAME function is used on the Raspberry Pi, so training and deployment match."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    s = size / max(w, h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    img = img.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    canvas.paste(img, ((size - nw) // 2, (size - nh) // 2))
    return canvas


class AnemiaClassifier:
    def __init__(self, model_path, config_path):
        with open(config_path) as f:
            self.cfg = json.load(f)
        self.size = self.cfg["img_size"]
        self.threshold = self.cfg["threshold"]
        self.interp = Interpreter(model_path=model_path, num_threads=4)  # Pi 5 has 4 cores
        self.interp.allocate_tensors()
        self.inp = self.interp.get_input_details()[0]
        self.out = self.interp.get_output_details()[0]

    def predict(self, pil_img):
        x = np.asarray(letterbox(pil_img, self.size), dtype=np.float32)[None]  # pixels 0-255
        self.interp.set_tensor(self.inp["index"], x.astype(self.inp["dtype"]))
        t0 = time.perf_counter()
        self.interp.invoke()
        ms = (time.perf_counter() - t0) * 1000
        prob = float(np.squeeze(self.interp.get_tensor(self.out["index"])))
        label = "ANEMIC (possible)" if prob >= self.threshold else "NOT ANEMIC"
        return prob, label, ms


def capture_from_camera():
    from picamera2 import Picamera2
    cam = Picamera2()
    cam.configure(cam.create_still_configuration())
    cam.start()
    time.sleep(2)                         # let exposure / white balance settle
    img = cam.capture_image("main")       # PIL image with correct RGB order
    cam.stop()
    return img.convert("RGB")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", help="path to a nail-bed photo")
    ap.add_argument("--camera", action="store_true", help="capture from the Pi camera")
    ap.add_argument("--model", default=os.path.join(HERE, "anemia_model_float32.tflite"))
    ap.add_argument("--config", default=os.path.join(HERE, "model_config.json"))
    ap.add_argument("--save", help="save the captured/input image here")
    a = ap.parse_args()

    if not a.image and not a.camera:
        ap.error("give --image PATH or --camera")
    img = capture_from_camera() if a.camera else Image.open(a.image)
    if a.save:
        img.save(a.save)

    clf = AnemiaClassifier(a.model, a.config)
    prob, label, ms = clf.predict(img)
    print(f"Result      : {label}")
    print(f"Probability : {prob:.3f}  (threshold {clf.threshold:.3f})")
    print(f"Inference   : {ms:.1f} ms")
    print("Screening only - confirm with a hemoglobin blood test.")


if __name__ == "__main__":
    main()
'''
with open(os.path.join(OUTPUT_DIR, "predict_pi.py"), "w") as f:
    f.write(PI_SCRIPT)

pkg_dir = "/content/anemia_pi_package" if IN_COLAB else os.path.join(OUTPUT_DIR, "pkg")
shutil.rmtree(pkg_dir, ignore_errors=True); os.makedirs(pkg_dir)
for fn in ["anemia_model_float32.tflite", "anemia_model_int8.tflite", "model_config.json", "predict_pi.py"]:
    if os.path.exists(os.path.join(OUTPUT_DIR, fn)):
        shutil.copy(os.path.join(OUTPUT_DIR, fn), pkg_dir)
zip_path = shutil.make_archive(os.path.join(os.path.dirname(pkg_dir), "anemia_pi_package"), "zip", pkg_dir)
print("Package:", zip_path)

if SAVE_TO_DRIVE and os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    shutil.copy(zip_path, DRIVE_OUTPUT_DIR)
    shutil.copy(os.path.join(OUTPUT_DIR, "anemia_model.keras"), DRIVE_OUTPUT_DIR)   # full model, for retraining later
    print("Copied to Google Drive:", DRIVE_OUTPUT_DIR)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## Step 11 (optional) — Try the model on your own photo
Upload a nail-bed photo and see the prediction (uses the exact same pre-processing as the Pi).

In [ ]:
if IN_COLAB:
    from google.colab import files
    up = files.upload()
    for fn in up:
        with Image.open(fn) as im:
            img = letterbox(im, IMG_SIZE)
        p = float(model.predict(np.asarray(img, dtype=np.float32)[None], verbose=0)[0, 0])
        plt.imshow(img); plt.axis("off")
        plt.title(f"{'ANEMIC (possible)' if p >= THRESHOLD else 'NOT ANEMIC'}   p={p:.2f}  (threshold {THRESHOLD:.2f})"); plt.show()

## Tips to make the model better
1. **Data matters more than code.** Aim for hundreds of *people* (not just images) per class, labelled from a real **hemoglobin blood test** (WHO: Hb < 12 g/dL women, < 13 g/dL men, < 11 g/dL children under 5).
2. **Cover every skin tone** in *both* classes. If all anemic photos come from darker-skinned people and all healthy ones from lighter-skinned people, the model will learn skin tone instead of anemia. Check the fairness table.
3. **Same capture setup for training and the Pi.** Colour is the whole signal, so use a fixed enclosure / finger rest with the same LED light and camera distance. Ideally collect your own training photos *with the Pi camera* in that enclosure.
4. **Crop to the nail.** Photos that are mostly nail bed work much better than whole hands.
5. **Remove nail polish**, and avoid photos right after cold exposure or pressing the finger (both change nail colour).